In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path
import pickle
from tabnanny import verbose
import spikeinterface as si
import numpy as np
from spikeinterface.core import get_template_extremum_channel
import scipy.spatial.distance
from scipy.sparse.csgraph import connected_components
import pickle
import pandas as pd
from utils_clique import (
    CliqueInfo,
    build_shank_cliques,
    neuron_inf_dict_to_dataframe,
    get_recording_clique,
    filter_neuron_inf_by_clique,
    filter_gt_detect_array_by_clique,
    prepare_training_data,
    train_autosort_model,
    build_sliding_cliques,
    process_unified_sorting,
    process_segment_data
)

files = sorted(os.listdir("/media/ubuntu/sda/data/mouse6/ns4/natural_image"))
recording_list = []
for file in files:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/{file}')
    recording_recorded = recording_raw.remove_channels(["98", '31', '32'])
    recording_list.append(recording_recorded)


probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = si.concatenate_recordings(recording_list)
recording_recorded = recording_recorded.set_probegroup(probe_30channel)

recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_f, freq=60)

print(recording_f)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_cmr = recording_cmr.rename_channels(['A-000', 'A-001', 'A-002', 'A-003', 'A-004',
                               'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
                               'A-0010', 'A-011', 'A-012', 'A-013', 'A-014',
                               'A-015', 'A-016', 'A-017', 'A-018', 'A-019',
                               'A-020', 'A-021', 'A-022', 'A-023', 'A-024',
                               'A-025', 'A-026', 'A-027', 'A-028', 'A-029'])
print(recording_cmr)


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BandpassFilterRecording: 30 channels - 10000.0Hz - 1 segments - 437,380,586 samples 
                         43,738.06s (12.15 hours) - int16 dtype - 24.44 GiB
ChannelSliceRecording: 30 channels - 10000.0Hz - 1 segments - 437,380,586 samples 
                       43,738.06s (12.15 hours) - int16 dtype - 24.44 GiB


In [2]:
recording_preprocessed = recording_cmr.save(format="binary", n_jobs = 30)   


Use cache_folder=/tmp/spikeinterface_cache/tmpuks8ryyx/08NVXNB3
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=17.17 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 43739/43739 [05:52<00:00, 124.10it/s]


In [ ]:
output_folder = '/media/ubuntu/sda/mouse_test/sorted/recordings_30_channel_12_months_mouse6_natim_full'

probe = recording_cmr.get_probe()
probe.set_contact_ids(recording_cmr.channel_ids)

# 对于30通道，创建一个包含所有通道的clique
probe_df = probe.to_dataframe()
all_channel_ids = probe_df['contact_ids'].astype(str).tolist()
cliques = [
    CliqueInfo(
        clique_id=0,
        device_channel_indices=list(range(len(all_channel_ids))),
        contact_ids=all_channel_ids,
        center=(probe_df['x'].mean(), probe_df['y'].mean())
    )
]

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'method': 'single_clique_all_channels',
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

In [ ]:
combined_output_base = output_folder

# 调用封装好的函数处理统一sorting结果
neuron_inf_all, gt_detect_array_all, analyzer = process_unified_sorting(
    recording_cmr=recording_cmr,
    output_folder=output_folder,
    distance_threshold=10.0,
    similarity_threshold=0.95,
    n_jobs=20,
    verbose=True
)

segment_neuron_inf, segment_gt_detect = process_segment_data(
    neuron_inf_all=neuron_inf_all,
    gt_detect_array_all=gt_detect_array_all,
    recording_list=recording_list,
    files=files,
    output_folder=output_folder + '/segments',
    firing_rate_threshold=0.5,
    verbose=True
)
